# GDELT CACHE

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from google.cloud import bigquery
import pydata_google_auth

SCOPES = ['https://www.googleapis.com/auth/bigquery']
credentials = pydata_google_auth.get_user_credentials(
    SCOPES,
    auth_local_webserver=True,
)

PROJECT_ID = 'gdelt-thesis-487419'
START_DATE = '20150401'
END_DATE   = '20260401'

client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
print(f'BigQuery client ready. Project: {PROJECT_ID}')
print(f'Sample period: {START_DATE} → {END_DATE}')

BigQuery client ready. Project: gdelt-thesis-487419
Sample period: 20150401 → 20260401


In [2]:
def regex_union(terms):
    return r'(?:' + '|'.join(
        re.escape(t.lower()).replace(r'\ ', r'[\s\-_]+') for t in terms
    ) + r')'

SEARCH_BLOB = ("LOWER(CONCAT("
    "IFNULL(V2Themes,''),' | ',IFNULL(V2Organizations,''),' | ',"
    "IFNULL(V2Persons,''),' | ',IFNULL(AllNames,''),' | ',"
    "IFNULL(DocumentIdentifier,'')))")

KEYWORD_CONCEPTS = {
    'neodymium':      ['neodymium'],
    'praseodymium':   ['praseodymium'],
    'dysprosium':     ['dysprosium'],
    'terbium':        ['terbium'],
    'ndpr':           ['ndpr'],
    'ndfeb':          ['ndfeb'],
    'lanthanum':      ['lanthanum'],
    'cerium':         ['cerium'],
    'samarium':       ['samarium'],
    'europium':       ['europium'],
    'gadolinium':     ['gadolinium'],
    'yttrium':        ['yttrium'],
    'rare_earth':     ['rare earth', 'rare earths', 'rare-earth', 'rare-earths',
                       'rare earth element', 'rare earth elements',
                       'rare-earth element', 'rare-earth elements',
                       'rare earth magnet', 'rare earth magnets',
                       'rare-earth magnet', 'rare-earth magnets',
                       'rare earth metal', 'rare earth metals',
                       'rare-earth metal', 'rare-earth metals'],
    'lynas':          ['lynas', 'lynas rare earths', 'lynas corporation'],
    'mp_materials':   ['mp materials'],
    'jl_mag':         ['jl mag', 'jl mag rare-earth'],
    'arafura':        ['arafura resources', 'arafura rare earths'],
    'china_northern': ['china northern rare earth'],
    'remx':           ['remx'],
}

THEMATIC_CONCEPTS = {
    'china':         ['china', 'beijing', 'chinese', "people's republic"],
    'export_ctrl':   ['export control', 'export ban', 'export restriction',
                      'export quota', 'export licence', 'export license',
                      'export curb', 'trade restriction'],
    'ev':            ['electric vehicle', 'electric vehicles', 'ev battery',
                      'ev motor', 'permanent magnet motor', 'traction motor',
                      'wind turbine', 'wind farm'],
    'supply_chain':  ['supply chain', 'supply disruption', 'supply shock',
                      'critical mineral', 'critical material',
                      'strategic mineral', 'strategic material'],
}

def build_filter(concepts, n_keywords, search_blob):
    return "(" + " + ".join(
        f"CAST(REGEXP_CONTAINS({search_blob}, r'\\b{regex_union(group)}\\b') AS INT64)"
        for group in concepts.values()
    ) + f") >= {n_keywords}"

keyword_filter_broad = build_filter(KEYWORD_CONCEPTS, n_keywords=1,
                                    search_blob=SEARCH_BLOB)

concept_selects = [
    f"REGEXP_CONTAINS({SEARCH_BLOB}, r'\\b{regex_union(group)}\\b') AS has_{name}"
    for name, group in KEYWORD_CONCEPTS.items()
]
thematic_selects = [
    f"REGEXP_CONTAINS({SEARCH_BLOB}, r'\\b{regex_union(group)}\\b') AS has_{name}"
    for name, group in THEMATIC_CONCEPTS.items()
]
all_flag_selects_sql = ",\n    ".join(concept_selects + thematic_selects)

print(f'REE concept flags : {len(KEYWORD_CONCEPTS)}')
print(f'Thematic flags    : {len(THEMATIC_CONCEPTS)}')
print(f'Broad filter ready.')

REE concept flags : 19
Thematic flags    : 4
Broad filter ready.


In [3]:
def regex_union(terms):
    return r'(?:' + '|'.join(
        re.escape(t.lower()).replace(r'\ ', r'[\s\-_]+') for t in terms
    ) + r')'

SEARCH_BLOB = ("LOWER(CONCAT("
    "IFNULL(V2Themes,''),' | ',IFNULL(V2Organizations,''),' | ',"
    "IFNULL(V2Persons,''),' | ',IFNULL(AllNames,''),' | ',"
    "IFNULL(DocumentIdentifier,'')))")

KEYWORD_CONCEPTS = {
    'neodymium':      ['neodymium'],
    'praseodymium':   ['praseodymium'],
    'dysprosium':     ['dysprosium'],
    'terbium':        ['terbium'],
    'ndpr':           ['ndpr'],
    'ndfeb':          ['ndfeb'],
    'lanthanum':      ['lanthanum'],
    'cerium':         ['cerium'],
    'samarium':       ['samarium'],
    'europium':       ['europium'],
    'gadolinium':     ['gadolinium'],
    'yttrium':        ['yttrium'],
    'rare_earth':     ['rare earth', 'rare earths', 'rare-earth', 'rare-earths',
                       'rare earth element', 'rare earth elements',
                       'rare-earth element', 'rare-earth elements',
                       'rare earth magnet', 'rare earth magnets',
                       'rare-earth magnet', 'rare-earth magnets',
                       'rare earth metal', 'rare earth metals',
                       'rare-earth metal', 'rare-earth metals'],
    'lynas':          ['lynas', 'lynas rare earths', 'lynas corporation'],
    'mp_materials':   ['mp materials'],
    'jl_mag':         ['jl mag', 'jl mag rare-earth'],
    'arafura':        ['arafura resources', 'arafura rare earths'],
    'china_northern': ['china northern rare earth'],
    'remx':           ['remx'],
}

# Note: apostrophes removed from all terms — they break BigQuery regex literals
THEMATIC_CONCEPTS = {
    'china':         ['china', 'beijing', 'chinese', 'peoples republic'],
    'export_ctrl':   ['export control', 'export ban', 'export restriction',
                      'export quota', 'export licence', 'export license',
                      'export curb', 'trade restriction'],
    'ev':            ['electric vehicle', 'electric vehicles', 'ev battery',
                      'ev motor', 'permanent magnet motor', 'traction motor',
                      'wind turbine', 'wind farm'],
    'supply_chain':  ['supply chain', 'supply disruption', 'supply shock',
                      'critical mineral', 'critical material',
                      'strategic mineral', 'strategic material'],
}

def build_filter(concepts, n_keywords, search_blob):
    return "(" + " + ".join(
        f"CAST(REGEXP_CONTAINS({search_blob}, r'\\b{regex_union(group)}\\b') AS INT64)"
        for group in concepts.values()
    ) + f") >= {n_keywords}"

keyword_filter_broad = build_filter(KEYWORD_CONCEPTS, n_keywords=1,
                                    search_blob=SEARCH_BLOB)

concept_selects = [
    f"REGEXP_CONTAINS({SEARCH_BLOB}, r'\\b{regex_union(group)}\\b') AS has_{name}"
    for name, group in KEYWORD_CONCEPTS.items()
]
thematic_selects = [
    f"REGEXP_CONTAINS({SEARCH_BLOB}, r'\\b{regex_union(group)}\\b') AS has_{name}"
    for name, group in THEMATIC_CONCEPTS.items()
]
all_flag_selects_sql = ",\n    ".join(concept_selects + thematic_selects)

print(f'REE concept flags : {len(KEYWORD_CONCEPTS)}')
print(f'Thematic flags    : {len(THEMATIC_CONCEPTS)}')
print(f'Broad filter ready.')

REE concept flags : 19
Thematic flags    : 4
Broad filter ready.


In [ ]:
CACHE_QUERY = f"""
SELECT
    CAST(DATE AS STRING)                                          AS pub_datetime_str,
    SUBSTR(CAST(DATE AS STRING), 1, 8)                           AS date_str,
    DocumentIdentifier                                           AS url,
    NET.REG_DOMAIN(DocumentIdentifier)                           AS domain,
    SourceCollectionIdentifier                                   AS source_coll_id,
    SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(0)] AS FLOAT64)    AS tone,
    SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(1)] AS FLOAT64)    AS pos_score,
    SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(2)] AS FLOAT64)    AS neg_score,
    SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(3)] AS FLOAT64)    AS polarity,
    SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(6)] AS FLOAT64)    AS wc,
    V2Themes                                                     AS v2themes_raw,
    V2Locations                                                  AS v2locations_raw,
    V2Organizations                                              AS v2orgs_raw,
    {all_flag_selects_sql}
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE
    _PARTITIONTIME >= TIMESTAMP('{START_DATE[:4]}-{START_DATE[4:6]}-{START_DATE[6:]}')
    AND _PARTITIONTIME <= TIMESTAMP('{END_DATE[:4]}-{END_DATE[4:6]}-{END_DATE[6:]}')
    AND DATE BETWEEN {START_DATE}000000 AND {END_DATE}235959
    AND SourceCollectionIdentifier = 1
    AND {keyword_filter_broad}
"""

MAX_BYTES_BILLED = int(3.0 * 1024**4)
PRICE_USD_PER_TIB = 6.25

dry_run = client.query(
    CACHE_QUERY,
    job_config=bigquery.QueryJobConfig(
        dry_run=True,
        use_query_cache=False,
        maximum_bytes_billed=MAX_BYTES_BILLED,
    )
)

bytes_scanned = dry_run.total_bytes_processed
gb_scanned    = bytes_scanned / (1024**3)
tib_scanned   = bytes_scanned / (1024**4)
cost_upper    = tib_scanned * PRICE_USD_PER_TIB

print(f'Estimated scan : {gb_scanned:,.1f} GB ({tib_scanned:.4f} TiB)')
print(f'Estimated cost : ${cost_upper:,.2f} USD (before free tier)')

if tib_scanned <= 1.0:
    print('✓ Within the 1 TiB/month free tier (if not already consumed)')
else:
    print(f'! Exceeds free tier by {tib_scanned - 1.0:.2f} TiB')

In [5]:


real_config = bigquery.QueryJobConfig(
    use_query_cache=False,
    maximum_bytes_billed=MAX_BYTES_BILLED,
)

print('Executing query... (2–5 minutes)')
df_cache = client.query(CACHE_QUERY, job_config=real_config).to_dataframe()

# Parse full timestamp
df_cache['pub_datetime'] = pd.to_datetime(
    df_cache['pub_datetime_str'], format='%Y%m%d%H%M%S', errors='coerce'
)
df_cache = df_cache.drop(columns=['pub_datetime_str'])

print(f'\nDone. {len(df_cache):,} articles retrieved.')
print(f'Date range : {df_cache["date_str"].min()} → {df_cache["date_str"].max()}')
print(f'Memory     : {df_cache.memory_usage(deep=True).sum() / (1024**2):.1f} MB')
print(f'\nSource collection breakdown:')
print(df_cache['source_coll_id'].value_counts().to_string())
print(f'Domain nulls : {df_cache["domain"].isna().sum():,}')
print(f'WC nulls     : {df_cache["wc"].isna().sum():,}')

OUT_PATH = Path('Sentiment data') / 'SentimentData_Aleix_REMX_v2.parquet'
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_cache.to_parquet(OUT_PATH, index=False)

size_mb = OUT_PATH.stat().st_size / (1024**2)
print(f'\nSaved → {OUT_PATH}  ({size_mb:.1f} MB, {len(df_cache):,} rows)')

NameError: name 'MAX_BYTES_BILLED' is not defined